# MS3SEG VentiMorph-RelNet-v2.7 — 5-Fold Best-Point Summary

This notebook consolidates the selected validation results from the five supplied notebooks
into one reproducible summary. **Every number is read (scraped) from the source `.ipynb`
files' saved cell outputs — nothing is typed in.**

**Mapping used to match the handwritten 5-fold sheet:**
- **Fold-1** → `fold0-validation-eva` patient-level 3D validation summary.
- **Fold-2** → `fold1-training` best balanced-score epoch.
- **Fold-3** → `fold2-training` best balanced-score epoch.
- **Fold-4** → `fold3-training` best balanced-score epoch.
- **Fold-5** → `fold4-training` best balanced-score epoch.

The handwritten label **Mean F1** is retained for presentation. In the source notebooks this
value is the **mean foreground Dice** (`mean_patient_foreground_dice` for Fold-1 and
`val_mean_fg_dice` for Folds 2–5). Fold-1 is a 3D patient-level Dice; Folds 2–5 are the
training-loop (slice-level) validation Dice at the selected epoch.


In [1]:
import json
import re
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.precision", 6)
print("Libraries loaded successfully.")


Libraries loaded successfully.


## 1. Collect the selected points from all five notebooks

The values below are pulled from the outputs already present in the supplied notebooks by
parsing their JSON. No new model training or inference is performed.


In [2]:
# --- locate the five source notebooks -------------------------------------------------
SEARCH_DIRS = [
    Path.cwd(),
    Path.cwd() / "paper update" / "MS3SEG_VentiMorph_RelNet_V2",
    Path.cwd().parent,
    Path.home() / "Downloads",
    Path.home() / "Downloads" / "paper update" / "MS3SEG_VentiMorph_RelNet_V2",
    Path(".."), Path("../.."),
]

PATTERNS = {
    "fold0-validation-eva": "*fold0-validation-eva*.ipynb",
    "fold1-training":       "*fold1-training*.ipynb",
    "fold2-training":       "*fold2-training*.ipynb",
    "fold3-training":       "*fold3-training*.ipynb",
    "fold4-training":       "*fold4-training*.ipynb",
}

def find_notebook(pattern):
    for d in SEARCH_DIRS:
        hits = sorted(d.glob(pattern)) if d.exists() else []
        if hits:
            return hits[0].resolve()
    raise FileNotFoundError(f"could not find a notebook matching {pattern!r}")

NB = {name: find_notebook(pat) for name, pat in PATTERNS.items()}
for name, path in NB.items():
    print(f"{name:24s} -> {path}")


fold0-validation-eva     -> C:\Users\ASUS\Downloads\paper update\MS3SEG_VentiMorph_RelNet_V2\ms3seg-ventimorph-relnet-v2-7-fold0-validation-eva.ipynb
fold1-training           -> C:\Users\ASUS\Downloads\paper update\MS3SEG_VentiMorph_RelNet_V2\ms3seg-ventimorph-relnet-v2-7-fold1-training (1).ipynb
fold2-training           -> C:\Users\ASUS\Downloads\paper update\MS3SEG_VentiMorph_RelNet_V2\ms3seg-ventimorph-relnet-v2-7-fold2-training (1).ipynb
fold3-training           -> C:\Users\ASUS\Downloads\paper update\MS3SEG_VentiMorph_RelNet_V2\ms3seg-ventimorph-relnet-v2-7-fold3-training (1).ipynb
fold4-training           -> C:\Users\ASUS\Downloads\paper update\MS3SEG_VentiMorph_RelNet_V2\ms3seg-ventimorph-relnet-v2-7-fold4-training (1).ipynb


In [3]:
# --- read every text output out of a notebook -----------------------------------------
def iter_output_texts(nb_path):
    nb = json.loads(Path(nb_path).read_text(encoding="utf-8"))
    for cell in nb.get("cells", []):
        for out in cell.get("outputs", []):
            if out.get("output_type") == "stream":
                yield "".join(out.get("text", []))
            elif out.get("output_type") in ("execute_result", "display_data"):
                yield "".join(out.get("data", {}).get("text/plain", []))


# --- Fold-1: parse the patient-level 3D 'overall_summary' table -----------------------
EVAL_KEYS = {
    "mean_patient_foreground_dice": "Mean F1",
    "mean_abnormal_wmh_dice":       "abWMH",
    "mean_normal_wmh_dice":         "nWMH",
    "mean_ventricle_dice":          "Ventricle",
}

def scrape_eval_overall(nb_path):
    for text in iter_output_texts(nb_path):
        if "mean_abnormal_wmh_dice" in text and "mean_ventricle_dice" in text:
            vals = {}
            for raw, label in EVAL_KEYS.items():
                m = re.search(rf"{raw}\s+([0-9]*\.[0-9]+)", text)
                if m:
                    vals[label] = float(m.group(1))
            if len(vals) == 4:
                return vals, "Patient-level 3D validation summary"
    raise RuntimeError(f"overall_summary not found in {nb_path}")


# --- Folds 2-5: parse every 'best epoch' table, keep the max val_balanced_score -------
NEEDED = {"epoch", "val_dice_vent", "val_dice_nwmh", "val_dice_abwmh",
          "val_mean_fg_dice", "val_balanced_score"}

def scrape_best_balanced(nb_path):
    candidates = []
    for text in iter_output_texts(nb_path):
        rec = {}
        for line in text.splitlines():
            m = re.match(r"^([A-Za-z_][A-Za-z0-9_]*)\s+(-?[0-9]+\.?[0-9]*)\s*$", line)
            if m:
                rec[m.group(1)] = float(m.group(2))
        if NEEDED.issubset(rec):
            candidates.append(rec)
    if not candidates:
        raise RuntimeError(f"no best-epoch table found in {nb_path}")
    best = max(candidates, key=lambda r: r["val_balanced_score"])
    vals = {"Mean F1": best["val_mean_fg_dice"], "abWMH": best["val_dice_abwmh"],
            "nWMH": best["val_dice_nwmh"], "Ventricle": best["val_dice_vent"]}
    return vals, f"Best balanced-score epoch (epoch {int(best['epoch'])})"


In [4]:
# --- assemble the 5-fold table by scraping (replaces the old hand-typed DataFrame) ----
FOLD_MAP = [
    ("Fold-1", "fold0-validation-eva", scrape_eval_overall),
    ("Fold-2", "fold1-training",       scrape_best_balanced),
    ("Fold-3", "fold2-training",       scrape_best_balanced),
    ("Fold-4", "fold3-training",       scrape_best_balanced),
    ("Fold-5", "fold4-training",       scrape_best_balanced),
]

rows = []
for fold, src, scraper in FOLD_MAP:
    vals, selection = scraper(NB[src])
    print(f"{fold}: scraped {vals} from {src}")
    rows.append({"Fold": fold, "Source notebook": src, "Selection": selection, **vals})

fold_results = pd.DataFrame(rows)[
    ["Fold", "Source notebook", "Selection", "Mean F1", "abWMH", "nWMH", "Ventricle"]
]
fold_results


Fold-1: scraped {'Mean F1': 0.755216, 'abWMH': 0.777938, 'nWMH': 0.63612, 'Ventricle': 0.851588} from fold0-validation-eva
Fold-2: scraped {'Mean F1': 0.739136, 'abWMH': 0.753907, 'nWMH': 0.62691, 'Ventricle': 0.836591} from fold1-training
Fold-3: scraped {'Mean F1': 0.745058, 'abWMH': 0.744385, 'nWMH': 0.640466, 'Ventricle': 0.850324} from fold2-training
Fold-4: scraped {'Mean F1': 0.762568, 'abWMH': 0.789882, 'nWMH': 0.642706, 'Ventricle': 0.855117} from fold3-training
Fold-5: scraped {'Mean F1': 0.745067, 'abWMH': 0.767844, 'nWMH': 0.616333, 'Ventricle': 0.851024} from fold4-training


,Fold,Source notebook,Selection,Mean F1,abWMH,nWMH,Ventricle
0,Fold-1,fold0-validation-eva,Patient-level 3D validation summary,0.755216,0.777938,0.636120,0.851588
1,Fold-2,fold1-training,Best balanced-score epoch (epoch 19),0.739136,0.753907,0.626910,0.836591
2,Fold-3,fold2-training,Best balanced-score epoch (epoch 58),0.745058,0.744385,0.640466,0.850324
3,Fold-4,fold3-training,Best balanced-score epoch (epoch 33),0.762568,0.789882,0.642706,0.855117
4,Fold-5,fold4-training,Best balanced-score epoch (epoch 43),0.745067,0.767844,0.616333,0.851024


## 2. Compact output matching the handwritten result sheet

In [5]:
metric_cols = ["Mean F1", "abWMH", "nWMH", "Ventricle"]

print("=" * 76)
print("MS3SEG VentiMorph-RelNet-v2.7 | Selected 5-Fold Validation Results")
print("=" * 76)
print(f"{'Fold':<10}{'Mean F1':>14}{'abWMH':>14}{'nWMH':>14}{'Ventricle':>14}")
print("-" * 76)

for _, row in fold_results.iterrows():
    print(
        f"{row['Fold']:<10}"
        f"{row['Mean F1']:>14.6f}"
        f"{row['abWMH']:>14.6f}"
        f"{row['nWMH']:>14.6f}"
        f"{row['Ventricle']:>14.6f}"
    )

print("-" * 76)


MS3SEG VentiMorph-RelNet-v2.7 | Selected 5-Fold Validation Results
Fold             Mean F1         abWMH          nWMH     Ventricle
----------------------------------------------------------------------------
Fold-1          0.755216      0.777938      0.636120      0.851588
Fold-2          0.739136      0.753907      0.626910      0.836591
Fold-3          0.745058      0.744385      0.640466      0.850324
Fold-4          0.762568      0.789882      0.642706      0.855117
Fold-5          0.745067      0.767844      0.616333      0.851024
----------------------------------------------------------------------------


## 3. Calculate the 5-fold sums and averages

In [6]:
metric_sums = fold_results[metric_cols].sum()
metric_means = fold_results[metric_cols].mean()

summary = pd.DataFrame({
    "Sum of 5 folds": metric_sums,
    "5-fold mean": metric_means,
    "5-fold std": fold_results[metric_cols].std(ddof=1),
})

summary


,Sum of 5 folds,5-fold mean,5-fold std
Mean F1,3.747045,0.749409,0.009357
abWMH,3.833956,0.766791,0.018211
nWMH,3.162535,0.632507,0.010878
Ventricle,4.244644,0.848929,0.007140


In [7]:
for metric in metric_cols:
    total = metric_sums[metric]
    avg = metric_means[metric]
    print(f"{metric:<10}: {total:.6f} / 5 = {avg:.7f}")


Mean F1   : 3.747045 / 5 = 0.7494090
abWMH     : 3.833956 / 5 = 0.7667912
nWMH      : 3.162535 / 5 = 0.6325070
Ventricle : 4.244644 / 5 = 0.8489288


## 4. Identify the best fold for each reported metric

In [8]:
best_rows = []
for metric in metric_cols:
    idx = fold_results[metric].idxmax()
    best_rows.append({
        "Metric": metric,
        "Best fold": fold_results.loc[idx, "Fold"],
        "Best value": fold_results.loc[idx, metric],
        "Source selection": fold_results.loc[idx, "Selection"],
    })

best_points = pd.DataFrame(best_rows)
best_points


,Metric,Best fold,Best value,Source selection
0,Mean F1,Fold-4,0.762568,Best balanced-score epoch (epoch 33)
1,abWMH,Fold-4,0.789882,Best balanced-score epoch (epoch 33)
2,nWMH,Fold-4,0.642706,Best balanced-score epoch (epoch 33)
3,Ventricle,Fold-4,0.855117,Best balanced-score epoch (epoch 33)


In [9]:
overall_idx = fold_results["Mean F1"].idxmax()
overall = fold_results.loc[overall_idx]

print(f"Best overall fold by Mean F1: {overall['Fold']}")
print(f"Mean F1   : {overall['Mean F1']:.6f}")
print(f"abWMH     : {overall['abWMH']:.6f}")
print(f"nWMH      : {overall['nWMH']:.6f}")
print(f"Ventricle : {overall['Ventricle']:.6f}")


Best overall fold by Mean F1: Fold-4
Mean F1   : 0.762568
abWMH     : 0.789882
nWMH      : 0.642706
Ventricle : 0.855117


## 5. Cross-check the 5-fold averages against the published Table 7

In [10]:
paper_table7 = {"Ventricle": 0.8489, "nWMH": 0.6325, "abWMH": 0.76679, "Mean F1": 0.7494}
check = pd.DataFrame({
    "computed 5-fold mean": metric_means,
    "paper Table 7": pd.Series(paper_table7),
})
check["difference"] = check["computed 5-fold mean"] - check["paper Table 7"]
check["match"] = check["difference"].abs() < 5e-4
check


,computed 5-fold mean,paper Table 7,difference,match
Mean F1,0.749409,0.74940,0.000009,True
Ventricle,0.848929,0.84890,0.000029,True
abWMH,0.766791,0.76679,0.000001,True
nWMH,0.632507,0.63250,0.000007,True


## 6. Export the consolidated table

Running this cell creates a CSV copy of the same 5-fold values for use in a report or spreadsheet.


In [11]:
csv_name = "MS3SEG_5Fold_Best_Points_Summary.csv"
fold_results.to_csv(csv_name, index=False)
print(f"Saved: {csv_name}")


Saved: MS3SEG_5Fold_Best_Points_Summary.csv


### Final 5-fold averages

- Mean F1 / mean foreground Dice: **0.7494090**
- abnormal WMH Dice: **0.7667912**
- normal WMH Dice: **0.6325070**
- ventricle Dice: **0.8489288**

These averages are calculated directly from the five scraped rows shown above and match the
paper's Table 7 (`0.7494 / 0.76679 / 0.6325 / 0.8489`).
